# Runtime Overhead Benchmark

Measures the wall-clock cost of adaptive-basis optimisation relative to
fixed-basis sparse simulation **under an identical gate-application kernel**.
Both simulators use hash-table deduplication (O(k) per gate); the overhead
reported here therefore reflects _only_ the cost of periodic RDM computation
and basis rotation — not any algorithmic asymmetry in gate application.

## Methodology

For each _(N, k)_ configuration:

1. **N_CIRCUITS distinct random brickwork circuits** are generated — one exact
   statevector is computed per circuit and reused.
2. For each circuit and each method (fixed / adaptive), the `simulate()` call
   is repeated **N_TIMING_REPS** times. The **median** of those repetitions is
   taken as the representative runtime for that (circuit, method) pair.
   _Median is robust to OS-scheduling spikes; minimum is not._
3. The **overhead ratio** for that circuit is `t_bass_median / t_fixed_median`.
4. Across the N_CIRCUITS circuits, the **geometric mean** of those ratios is
   reported as the headline overhead, with a **non-parametric bootstrap 95% CI**
   (percentile method, N_BOOT resamples). Geometric mean is appropriate because
   the ratio spans an order of magnitude and is multiplicatively distributed.

Fidelity and participation-ratio statistics are geometric means over the same
N_CIRCUITS independent circuit instances.


In [ ]:
from pathlib import Path
import csv, json, sys, time

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

THIS_DIR = REPO_ROOT / "tests"
FIGURE_DIR = THIS_DIR / "figures"
DATA_DIR = THIS_DIR / "results"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

from src.experimentation.runner import (
    make_brickwork,
    exact_statevector,
    fidelity_bass,
)
from src.benchmarking.fidelity import compute_fidelity, compute_participation_ratio
from src.simulation.bass_simulator import BASS
from src.simulation.simulator import FixedBasisSimulator
from src.visualization.style import mplstyle, polish_axes, save_figure

mplstyle(dpi=180)
mpl.rcParams.update(
    {
        "font.size": 8.0,
        "axes.labelsize": 8.0,
        "xtick.labelsize": 7.0,
        "ytick.labelsize": 7.0,
        "legend.fontsize": 7.0,
        "lines.linewidth": 1.35,
        "lines.markersize": 4.0,
        "axes.linewidth": 0.8,
    }
)

print(f"Repository root : {REPO_ROOT}")
print(f"Figure output   : {FIGURE_DIR}")
print(f"Data output     : {DATA_DIR}")

In [ ]:
# ── Benchmark configuration ───────────────────────────────────────────────────

N_VALUES = [12, 14, 16, 18, 20]
K_VALUES = [500, 1_000, 2_000, 5_000, 10_000, 20_000]
DEPTH = 6  # brickwork circuit depth

# Timing design
N_CIRCUITS = 10  # independent random circuit instances per (N, k)
N_TIMING_REPS = 10  # repetitions per (circuit, method); median taken → robust to jitter
N_BOOT = 4_000  # bootstrap resamples for 95% CI on geometric mean
CI_LEVEL = 95  # confidence level (%)

SEED_BASE = 20260520
FORCE_RERUN = False
ADAPTIVE_KWARGS = dict(optimize_every=5, buffer_factor=1)

# ── Visualization ─────────────────────────────────────────────────────────────

METHOD_LABELS = {"fixed": "Fixed basis", "adaptive": "Adaptive basis (BASS)"}

COLORS = {
    500: "#0C5DA5",
    1_000: "#9D31BE",
    2_000: "#DDAA33",
    5_000: "#BB5566",
    10_000: "#228833",
    20_000: "#AA3377",
}


# PRZ ~ 2^(0.7 N) for brickwork L=6
# Used only to annotate undersampled / oversampled regime in the summary table.
def prz_estimate(N):
    return 2 ** (0.70 * N)

In [ ]:
# ── Timing utilities ──────────────────────────────────────────────────────────


def _time_fixed(circ, N, k, n_reps):
    """
    Time FixedBasisSimulator.simulate() over n_reps repetitions.

    Returns (median_s, final_state).  The simulator object is constructed once;
    simulate() reinitialises SparseState at the top of every call, so each
    repetition is a fully independent simulation with identical input.
    Constructor overhead is excluded from timing.
    """
    sim = FixedBasisSimulator(N, k, verbose=False)
    times = np.empty(n_reps)
    state = None
    for i in range(n_reps):
        t0 = time.perf_counter()
        state = sim.simulate(circ)
        times[i] = time.perf_counter() - t0
    return float(np.median(times)), state


def _time_bass(circ, N, k, bass_kwargs, n_reps):
    """
    Time BASS.simulate() over n_reps repetitions.

    BASS.simulate() reinitialises self.U and _gamma_running at the top of every
    call, so each repetition is a fully independent adaptive simulation.
    Constructor overhead is excluded from timing.
    """
    sim = BASS(N, k, **bass_kwargs)
    times = np.empty(n_reps)
    state = None
    for i in range(n_reps):
        t0 = time.perf_counter()
        state = sim.simulate(circ)
        times[i] = time.perf_counter() - t0
    return float(np.median(times)), state, sim


def bootstrap_geomean_ci(values, n_boot=N_BOOT, ci=CI_LEVEL, seed=0):
    """
    Non-parametric bootstrap percentile CI for the geometric mean.

    Resamples `values` with replacement n_boot times.  Returns
    (geometric_mean, ci_lo, ci_hi) where (ci_lo, ci_hi) is the
    (alpha/2, 1-alpha/2) percentile interval on exp(mean(log(values))).

    Uses a percentile (not BCa) interval; sufficient for n>=10.
    """
    arr = np.asarray(values, dtype=float)
    arr = arr[arr > 0]
    if len(arr) == 0:
        return np.nan, np.nan, np.nan
    rng_b = np.random.default_rng(seed)
    log_arr = np.log(arr)
    boot_log_means = np.array(
        [
            np.mean(rng_b.choice(log_arr, size=len(log_arr), replace=True))
            for _ in range(n_boot)
        ]
    )
    alpha = (100 - ci) / 2.0
    return (
        float(np.exp(np.mean(log_arr))),
        float(np.exp(np.percentile(boot_log_means, alpha))),
        float(np.exp(np.percentile(boot_log_means, 100 - alpha))),
    )


def _gm(arr):
    """Geometric mean of positive entries."""
    a = np.asarray(arr, dtype=float)
    a = a[a > 0]
    return float(np.exp(np.mean(np.log(a)))) if len(a) > 0 else np.nan


def warmup():
    """
    Trigger Numba JIT compilation for all kernels used in the benchmark.
    Two passes: first compiles, second confirms steady-state.
    """
    for n, k in [(8, 32), (10, 128)]:
        rng = np.random.default_rng(SEED_BASE)
        circ = make_brickwork(n, 3, rng)
        FixedBasisSimulator(n, k, verbose=False).simulate(circ)
        BASS(n, k, **ADAPTIVE_KWARGS).simulate(circ)
    print("Warmup complete — all Numba kernels compiled.")


# ── Main benchmark ────────────────────────────────────────────────────────────


def cache_paths():
    return (
        DATA_DIR / "runtime_overhead.csv",
        DATA_DIR / "runtime_overhead_raw.npz",
        DATA_DIR / "runtime_overhead_metadata.json",
    )


def run_benchmark():
    """
    Core timing loop.

    Design
    ------
    Outer loop : N_CIRCUITS independent circuits per N.
    Inner loop : N_TIMING_REPS repetitions per (circuit, method) → median.

    This decouples two sources of variance:
    * Circuit-to-circuit variance (entanglement structure, truncation activation)
    → captured across N_CIRCUITS circuits.
    * Per-run timing noise (OS jitter, cache state)
    → suppressed by taking the per-circuit median over N_TIMING_REPS reps.

    Overhead per circuit = t_bass_median / t_fixed_median.
    Aggregate over circuits: geometric mean + bootstrap 95% CI.
    """
    warmup()

    rows = []

    # Raw per-circuit arrays stored for bootstrap and diagnostics.
    # Shape: (len(N_VALUES), len(K_VALUES), N_CIRCUITS)
    raw_overheads = np.full((len(N_VALUES), len(K_VALUES), N_CIRCUITS), np.nan)
    raw_t_fixed = np.full_like(raw_overheads, np.nan)
    raw_t_bass = np.full_like(raw_overheads, np.nan)
    raw_f_fixed = np.full_like(raw_overheads, np.nan)
    raw_f_bass = np.full_like(raw_overheads, np.nan)
    raw_pr_fixed = np.full_like(raw_overheads, np.nan)
    raw_pr_bass = np.full_like(raw_overheads, np.nan)

    hdr = (
        f"{'N':>4} {'k':>7}  "
        f"{'gm_ovhd':>8} {'ci_lo':>7} {'ci_hi':>7}  "
        f"{'med_ovhd':>9} {'iqr25':>7} {'iqr75':>7}  "
        f"{'gm_t_fix':>9} {'gm_t_bas':>9}  "
        f"{'gm_F_fix':>9} {'gm_F_bas':>9}  "
        f"{'regime':>12}"
    )
    print(hdr)
    print("-" * len(hdr))

    for n_idx, N in enumerate(N_VALUES):

        # ── Generate N_CIRCUITS distinct circuits for this N ──
        circuits = []
        exact_svs = []
        for c in range(N_CIRCUITS):
            seed_c = SEED_BASE + N * 10_000 + c
            circ = make_brickwork(N, DEPTH, np.random.default_rng(seed_c))
            circuits.append(circ)
            exact_svs.append(exact_statevector(N, circ))
        gate_count = len(circuits[0])

        for k_idx, k in enumerate(K_VALUES):

            ovhd_arr = np.empty(N_CIRCUITS)
            tf_arr = np.empty(N_CIRCUITS)
            tb_arr = np.empty(N_CIRCUITS)
            ff_arr = np.empty(N_CIRCUITS)
            fb_arr = np.empty(N_CIRCUITS)
            prf_arr = np.empty(N_CIRCUITS)
            prb_arr = np.empty(N_CIRCUITS)

            for c, (circ, esv) in enumerate(zip(circuits, exact_svs)):

                # ── Time fixed-basis ──
                t_fix, fs = _time_fixed(circ, N, k, N_TIMING_REPS)

                # ── Time BASS ──
                t_bas, bs, bass_sim = _time_bass(
                    circ, N, k, ADAPTIVE_KWARGS, N_TIMING_REPS
                )

                # ── Metrics (from last timing-rep result — deterministic) ──
                ff = compute_fidelity(fs, esv)
                fb = fidelity_bass(bass_sim, bs, esv)
                prf = compute_participation_ratio(fs)
                prb = compute_participation_ratio(bs)

                ovhd_arr[c] = t_bas / t_fix
                tf_arr[c] = t_fix
                tb_arr[c] = t_bas
                ff_arr[c] = ff
                fb_arr[c] = fb
                prf_arr[c] = prf
                prb_arr[c] = prb

            # Store raw arrays
            raw_overheads[n_idx, k_idx] = ovhd_arr
            raw_t_fixed[n_idx, k_idx] = tf_arr
            raw_t_bass[n_idx, k_idx] = tb_arr
            raw_f_fixed[n_idx, k_idx] = ff_arr
            raw_f_bass[n_idx, k_idx] = fb_arr
            raw_pr_fixed[n_idx, k_idx] = prf_arr
            raw_pr_bass[n_idx, k_idx] = prb_arr

            # ── Aggregate ──
            gm_ovhd, ci_lo, ci_hi = bootstrap_geomean_ci(ovhd_arr)
            med_ovhd = float(np.median(ovhd_arr))
            iqr_lo = float(np.percentile(ovhd_arr, 25))
            iqr_hi = float(np.percentile(ovhd_arr, 75))

            gm_tf, _, _ = bootstrap_geomean_ci(tf_arr)
            gm_tb, _, _ = bootstrap_geomean_ci(tb_arr)

            regime = "under" if k < prz_estimate(N) else "over"

            row = dict(
                N=N,
                k=k,
                gm_overhead=gm_ovhd,
                ci_lo=ci_lo,
                ci_hi=ci_hi,
                median_overhead=med_ovhd,
                iqr_lo=iqr_lo,
                iqr_hi=iqr_hi,
                gm_t_fixed_s=gm_tf,
                gm_t_bass_s=gm_tb,
                gm_f_fixed=_gm(ff_arr),
                gm_f_bass=_gm(fb_arr),
                gm_pr_fixed=_gm(prf_arr),
                gm_pr_bass=_gm(prb_arr),
                gate_count=gate_count,
                n_circuits=N_CIRCUITS,
                n_timing_reps=N_TIMING_REPS,
                regime=regime,
            )
            rows.append(row)

            print(
                f"{N:>4} {k:>7}  "
                f"{gm_ovhd:>8.3f} {ci_lo:>7.3f} {ci_hi:>7.3f}  "
                f"{med_ovhd:>9.3f} {iqr_lo:>7.3f} {iqr_hi:>7.3f}  "
                f"{gm_tf:>9.5f} {gm_tb:>9.5f}  "
                f"{_gm(ff_arr):>9.3e} {_gm(fb_arr):>9.3e}  "
                f"{regime:>12}"
            )

    return rows, dict(
        raw_overheads=raw_overheads,
        raw_t_fixed=raw_t_fixed,
        raw_t_bass=raw_t_bass,
        raw_f_fixed=raw_f_fixed,
        raw_f_bass=raw_f_bass,
        raw_pr_fixed=raw_pr_fixed,
        raw_pr_bass=raw_pr_bass,
    )


def run_or_load():
    csv_path, npz_path, meta_path = cache_paths()

    expected_meta = dict(
        n_values=N_VALUES,
        k_values=K_VALUES,
        depth=DEPTH,
        n_circuits=N_CIRCUITS,
        n_timing_reps=N_TIMING_REPS,
        n_boot=N_BOOT,
        ci_level=CI_LEVEL,
        seed_base=SEED_BASE,
        adaptive_kwargs=ADAPTIVE_KWARGS,
    )

    if (
        not FORCE_RERUN
        and csv_path.exists()
        and npz_path.exists()
        and meta_path.exists()
    ):
        stored = json.loads(meta_path.read_text(encoding="utf-8"))
        if all(stored.get(k) == v for k, v in expected_meta.items()):
            print(f"Loading cached results: {csv_path}")
            rows = []
            with csv_path.open(newline="", encoding="utf-8") as fh:
                for r in csv.DictReader(fh):
                    rows.append(
                        {
                            k: (
                                int(v)
                                if k
                                in (
                                    "N",
                                    "k",
                                    "gate_count",
                                    "n_circuits",
                                    "n_timing_reps",
                                )
                                else (v if k == "regime" else float(v))
                            )
                            for k, v in r.items()
                        }
                    )
            raw = dict(np.load(npz_path))
            return rows, raw
        print("Cache metadata mismatch — recomputing.")

    rows, raw = run_benchmark()

    # ── Persist ──
    meta_path.write_text(json.dumps(expected_meta, indent=2), encoding="utf-8")

    with csv_path.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    np.savez_compressed(npz_path, **raw)

    print(f"\nSaved {csv_path}")
    print(f"Saved {npz_path}")
    print(f"Saved {meta_path}")
    return rows, raw


rows, raw = run_or_load()

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────


def rows_for_k(k_val):
    return sorted([r for r in rows if r["k"] == k_val], key=lambda r: r["N"])


def rows_for_regime(regime):
    return [r for r in rows if r["regime"] == regime]


all_gm = np.array([r["gm_overhead"] for r in rows])
all_ci_lo = np.array([r["ci_lo"] for r in rows])
all_ci_hi = np.array([r["ci_hi"] for r in rows])
all_med = np.array([r["median_overhead"] for r in rows])

under = rows_for_regime("under")
over = rows_for_regime("over")


def regime_summary(label, subset):
    if not subset:
        print(f"  {label}: no data")
        return
    gm_vals = np.array([r["gm_overhead"] for r in subset])
    med_vals = np.array([r["median_overhead"] for r in subset])
    gm_of_gm, gm_ci_lo, gm_ci_hi = bootstrap_geomean_ci(gm_vals)
    print(f"  {label} ({len(subset)} configs):")
    print(
        f"    geom-mean of geom-mean overheads : {gm_of_gm:.3f}x  "
        f"[{gm_ci_lo:.3f}, {gm_ci_hi:.3f}]  95% bootstrap CI"
    )
    print(
        f"    median of median overheads       : {np.median(med_vals):.3f}x  "
        f"IQR [{np.percentile(med_vals,25):.3f}, {np.percentile(med_vals,75):.3f}]"
    )
    print(
        f"    range of geom-mean overheads     : [{gm_vals.min():.3f}, {gm_vals.max():.3f}]"
    )


print("=" * 70)
print("OVERHEAD SUMMARY  (geometric mean ± 95% bootstrap CI over circuits)")
print("=" * 70)
regime_summary("ALL configs", rows)
print()
regime_summary(f"Undersampled (k < PRZ ≈ 2^0.7N)", under)
print()
regime_summary(f"Oversampled  (k ≥ PRZ ≈ 2^0.7N)", over)

print()
print("=" * 70)
print("FULL RESULTS TABLE")
print("=" * 70)
hdr = (
    f"{'N':>4} {'k':>7}  {'gm_ovhd':>8} {'ci_lo':>7} {'ci_hi':>7}  "
    f"{'med_ovhd':>9} {'iqr25':>7} {'iqr75':>7}  "
    f"{'gm_F_fix':>10} {'gm_F_bas':>10}  {'regime':>10}"
)
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(
        f"{r['N']:>4} {r['k']:>7}  "
        f"{r['gm_overhead']:>8.3f} {r['ci_lo']:>7.3f} {r['ci_hi']:>7.3f}  "
        f"{r['median_overhead']:>9.3f} {r['iqr_lo']:>7.3f} {r['iqr_hi']:>7.3f}  "
        f"{r['gm_f_fixed']:>10.3e} {r['gm_f_bass']:>10.3e}  "
        f"{r['regime']:>10}"
    )

In [ ]:
# ── LaTeX table (Table V) ────────────────────────────────────────
#
# Reports geometric-mean runtime, geometric-mean overhead with bootstrap 95% CI,
# and geometric-mean fidelity.  Replaces the minimum-runtime table in the paper.


def fmt_s(sec):
    """Format seconds to 3 sig-figs with appropriate unit prefix."""
    if sec < 0.001:
        return f"{sec*1000:.2f}\\,ms"
    return f"{sec:.4f}\\,s"


def fmt_fid(f):
    if np.isnan(f) or f <= 0:
        return r"$<10^{-6}$"
    exp = int(np.floor(np.log10(f)))
    mant = f / 10**exp
    if abs(mant - 1.0) < 0.05:
        return rf"$10^{{{exp}}}$"
    return rf"${mant:.2f}\times10^{{{exp}}}$"


def fmt_ovhd(r):
    gm = r["gm_overhead"]
    lo = r["ci_lo"]
    hi = r["ci_hi"]
    # express CI as symmetric ± on the dominant side
    return rf"${gm:.2f}\times$ $[{lo:.2f},{hi:.2f}]$"


lines = []
lines.append(r"\begin{table*}[t]")
lines.append(r"\centering")
lines.append(r"\small")
lines.append(r"\caption{Runtime and fidelity for brickwork circuits")
lines.append(r"  (depth $L=6$, " + str(N_CIRCUITS) + r" independent circuit instances,")
lines.append(r"  " + str(N_TIMING_REPS) + r" timing repetitions per instance).  ")
lines.append(r"  $t_{\mathrm{fixed}}$ and $t_{\mathrm{BASS}}$ are geometric-mean")
lines.append(
    r"  wall-clock runtimes; overhead is the geometric mean of the per-circuit"
)
lines.append(
    r"  median-ratio $t_{\mathrm{BASS}}/t_{\mathrm{fixed}}$, with 95\% bootstrap"
)
lines.append(
    r"  confidence interval in brackets.  Fidelity values are geometric means over"
)
lines.append(r"  the same circuit instances.  Both simulators use identical hash-table")
lines.append(
    r"  gate-application kernels; overhead reflects only basis-optimisation cost.}"
)
lines.append(r"\label{tab:runtime}")
lines.append(r"\begin{tabular}{rr rr r rr}")
lines.append(r"\toprule")
lines.append(
    r"$N$ & $k$ & $t_{\mathrm{fixed}}$ & $t_{\mathrm{BASS}}$ "
    r"& Overhead [95\% CI] "
    r"& $F_{\mathrm{fixed}}$ & $F_{\mathrm{BASS}}$ \\"
)
lines.append(r"\midrule")

prev_N = None
for r in rows:
    if r["N"] != prev_N and prev_N is not None:
        lines.append(r"\addlinespace[2pt]")
    prev_N = r["N"]

    fid_ratio = (
        r["gm_f_bass"] / r["gm_f_fixed"] if r["gm_f_fixed"] > 0 else float("nan")
    )
    fid_ratio_str = rf"\;(${fid_ratio:.1f}\times$)" if not np.isnan(fid_ratio) else ""

    lines.append(
        f"  {r['N']} & {r['k']:,} "
        f"& {fmt_s(r['gm_t_fixed_s'])} "
        f"& {fmt_s(r['gm_t_bass_s'])} "
        f"& {fmt_ovhd(r)} "
        f"& {fmt_fid(r['gm_f_fixed'])} "
        f"& {fmt_fid(r['gm_f_bass'])}{fid_ratio_str} \\\\"
    )

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\end{table*}")

latex_str = "\n".join(lines)
print(latex_str)

latex_path = DATA_DIR / "table_runtime_overhead.tex"
latex_path.write_text(latex_str + "\n", encoding="utf-8")
print(f"\nLaTeX table saved to {latex_path}")